In [1]:
# ============================================
# Startup Cell: Mount Drive + Load Config + Verify Inputs
# ============================================

from google.colab import drive
drive.mount("/content/drive")

import os
import sys

# -------------------------------------------------
# Add project src directory to path
# -------------------------------------------------
SRC_DIR = "/content/drive/MyDrive/DIP_Project/src"
if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

# -------------------------------------------------
# Import project configuration
# -------------------------------------------------
from project_config import *

TRAIN_FEATURE_VECTORS_NORMALIZED_CSV = os.path.join(METADATA_DIR, TRAIN_NORMALIZED_FILENAME)

BASELINE_RESULTS_CSV = os.path.join(MODELS_DIR, "classifier_comparison_baseline.csv")
TUNED_RESULTS_CSV = os.path.join(MODELS_DIR, "classifier_comparison_tuned.csv")
BEST_CLASSIFIER_JSON = os.path.join(MODELS_DIR, "best_classifier_config.json")

print("Project configuration loaded.")
print()

# -------------------------------------------------
# Verify required input file
# -------------------------------------------------
print("Verifying required input file...")

if os.path.exists(TRAIN_FEATURE_VECTORS_NORMALIZED_CSV):
    print(f"Found: {TRAIN_FEATURE_VECTORS_NORMALIZED_CSV}")
else:
    raise FileNotFoundError(f"Missing required file: {TRAIN_FEATURE_VECTORS_NORMALIZED_CSV}")

print("\nStartup complete.")
print(f"Metadata directory: {METADATA_DIR}")
print(f"Models directory:   {MODELS_DIR}")



Mounted at /content/drive
Project configuration loaded.

Verifying required input file...
Found: /content/drive/MyDrive/DIP_Project/data/metadata/train_feature_vectors_normalized.csv

Startup complete.
Metadata directory: /content/drive/MyDrive/DIP_Project/data/metadata
Models directory:   /content/drive/MyDrive/DIP_Project/models


In [2]:
# ============================================================
# Cell 0 — Notebook Overview: Classifier Selection
# ============================================================

# PURPOSE:
# This notebook performs classifier selection to determine which
# scikit-learn classification algorithm is most effective for
# distinguishing AI-generated and real images using the previously
# extracted 25-dimensional Digital Image Processing (DIP) feature vectors.
#
# Multiple classifiers are evaluated using the same normalized
# training data so that classifier performance can be compared
# fairly and consistently. The strongest classifier candidates
# identified here will guide subsequent model selection and
# final independent test evaluation.

# INPUTS:
# - Normalized classifier-input dataset generated from:
#   06_Normalize_and_Prepare_Classifier_Input.ipynb
#
# Expected input file:
#   - train_feature_vectors_normalized.csv
#
# This dataset contains:
#   - metadata columns:
#       - filename
#       - class_label
#       - source_dataset
#       - subset
#   - 25 normalized DIP feature columns

# NOTE:
# All features have already been normalized using a scaler fit only
# on the training dataset in the previous notebook. No additional
# normalization should be applied here.

# ASSUMPTIONS:
# - Dataset collection, preprocessing, and train/test splitting are complete
# - DIP feature extraction is complete
# - Final normalized classifier-input dataset has been prepared
# - The independent test dataset exists but is not used in this notebook
# - Features are numerical and contain no missing values
# - Training labels are correctly aligned with feature rows
# - The 25-feature DIP vector is used as input to all classifiers
# - Paths and filenames are defined in project_config.py
# - scikit-learn is available in the current Python environment

# PROCESS OVERVIEW:
# This notebook uses scikit-learn to compare a set of classification
# algorithms on the same normalized DIP feature set. The workflow consists of:
#
# - loading and validating the normalized training dataset
# - separating metadata columns from feature columns
# - defining a set of candidate classifiers
# - evaluating baseline classifier performance using stratified k-fold cross-validation
# - ranking classifiers using multiple performance metrics
# - applying light hyperparameter tuning to top-performing classifiers
# - saving comparison results for reporting
# - recommending classifier(s) for subsequent final training and test evaluation

# OUTPUTS:
# - Ranked classifier comparison table (baseline results)
# - Identification of top-performing classifier candidates
# - Lightly tuned classifier results for top models
# - Saved CSV files:
#     - classifier_comparison_baseline.csv
#     - classifier_comparison_tuned.csv
# - Summary of recommended classifier(s) for subsequent notebooks

# These results support classifier choice for later notebooks
# and provide evidence for comparing classifier performance
# on the same DIP feature set.

# KEY DESIGN CHOICE:
# This notebook performs classifier selection only.
# It is intended to compare different classifier types
# (for example, Logistic Regression, SVM, Random Forest,
# Extra Trees, Gradient Boosting, and MLPClassifier)
# using the same normalized training features.
#
# Model evaluation is performed using stratified k-fold
# cross-validation on the training dataset.
#
# This notebook does NOT perform:
# - final full hyperparameter optimization
# - final independent test-set evaluation
# - replacement of the independent test procedure

# Those steps are performed in later notebooks.

# TERMINOLOGY NOTE:
# In this notebook, the primary comparison is across classifiers
# (algorithm types), not across fully optimized final models.
# Each classifier evaluated here produces a trained model instance,
# but the main purpose of the notebook is classifier selection.

# IMPORTANT NOTES:
# - Input features are already scaled from Notebook 06
# - All classifiers must use the same training data for fair comparison
# - The held-out test dataset must remain untouched until final evaluation
# - Metrics such as Accuracy, Precision, Recall, F1 Score,
#   and ROC-AUC are used for classifier comparison
# - Tree-based classifiers do not require scaling, but scaled inputs
#   are retained for consistency across all classifiers
# - Execution time for classifier comparison can be reduced by using
#   a High-RAM runtime configuration in Colab

# ============================================================
# CELL-BY-CELL STRUCTURE
# ============================================================

# Cell 1:
# - Import required libraries

# Cell 2:
# - Load normalized training dataset
# - Display shape and preview data

# Cell 3:
# - Perform sanity checks
#   - missing values
#   - class distribution
#   - expected feature count (25)

# Cell 4:
# - Separate metadata and feature columns
# - Build X_train and y_train
# - Encode class labels if required

# Cell 5:
# - Define candidate classifiers
# - Define scoring metrics

# Cell 6:
# - Run baseline classifier comparison using stratified k-fold cross-validation

# Cell 7:
# - Compile and rank classifier results

# Cell 8:
# - Apply light hyperparameter tuning to top classifiers

# Cell 9:
# - Save classifier comparison results

# Cell 10:
# - Summarize findings and recommend classifier(s)

print("Notebook overview loaded.")



Notebook overview loaded.


In [3]:
# ============================================
# Cell 1: Import Required Libraries
# ============================================

import os
import json
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV
from sklearn.preprocessing import LabelEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")



Libraries imported successfully.


In [4]:
# ============================================
# Cell 2: Load Normalized Training Data
# ============================================

print(f"Loading training data from: {TRAIN_FEATURE_VECTORS_NORMALIZED_CSV}")

df_train = pd.read_csv(TRAIN_FEATURE_VECTORS_NORMALIZED_CSV)

print("\nTraining dataset loaded successfully.")
print(f"Shape: {df_train.shape}")

print("\nFirst 5 rows:")
display(df_train.head())

print("\nColumn names:")
for col in df_train.columns:
    print(col)



Loading training data from: /content/drive/MyDrive/DIP_Project/data/metadata/train_feature_vectors_normalized.csv

Training dataset loaded successfully.
Shape: (14400, 29)

First 5 rows:


,filename,class_label,source_dataset,subset,Mean Gradient,Std Gradient,Max Gradient,Gradient Entropy,Edge Density,Orientation Mean,...,Patch Variance Std,Noise Residual Energy,Low Frequency Energy Ratio,High Frequency Energy Ratio,Radial Mean,Radial Std,Radial Entropy,Spectral Centroid,Spectral Bandwidth,Log Spectrum Std
0,rl_imgn_002320.png,rl,ImageNet_1K_256,train,0.973214,1.208264,0.913526,0.776979,0.735101,-0.207859,...,1.019527,1.071297,-0.189219,0.417247,0.448644,0.439427,-0.549996,-0.084929,0.290945,-1.254194
1,rl_coco_001397.png,rl,MS_COCO_2017,train,-0.450913,-0.225225,0.641767,-0.488332,-0.447341,-0.246587,...,0.509586,-0.330229,-0.264290,0.144170,-1.327122,-1.325683,1.397134,0.856623,0.808074,0.071118
2,rl_imgn_001958.png,rl,ImageNet_1K_256,train,0.605933,0.723094,0.717432,0.610900,0.160581,-0.468100,...,0.235488,0.851489,-0.544033,0.805334,-0.343146,-0.332461,-0.549996,-0.217414,0.432281,-1.341491
3,rl_coco_000800.png,rl,MS_COCO_2017,train,0.021388,0.357107,0.528482,-0.077406,0.331237,1.075664,...,0.395378,0.188882,0.518813,-0.349898,1.963337,1.956027,-0.549996,-0.448685,-0.628062,-1.111831
4,ai_mj_002892.png,ai,Midjourney,train,-0.212459,-0.661686,-0.209270,0.288772,-0.061089,-0.293840,...,-0.373574,-0.281104,0.607047,-0.556607,-0.011009,-0.013482,-0.549996,-0.185376,-0.470299,0.325939



Column names:
filename
class_label
source_dataset
subset
Mean Gradient
Std Gradient
Max Gradient
Gradient Entropy
Edge Density
Orientation Mean
Orientation Std
Orientation Entropy
Global Entropy
Local Entropy Mean
Local Entropy Std
Intensity Mean
Intensity Std
Laplacian Variance
Patch Variance Mean
Patch Variance Std
Noise Residual Energy
Low Frequency Energy Ratio
High Frequency Energy Ratio
Radial Mean
Radial Std
Radial Entropy
Spectral Centroid
Spectral Bandwidth
Log Spectrum Std


In [5]:
# ============================================
# Cell 3: Validate Training Data
# ============================================

print("Performing sanity checks...\n")

# --------------------------------------------
# Check for missing values
# --------------------------------------------
missing_values = df_train.isnull().sum().sum()
print(f"Total missing values: {missing_values}")

if missing_values == 0:
    print("No missing values detected.")
else:
    print("WARNING: Missing values detected.")

# --------------------------------------------
# Check class distribution
# --------------------------------------------
print("\nClass distribution:")
class_counts = df_train["class_label"].value_counts()
print(class_counts)

# --------------------------------------------
# Verify metadata columns
# --------------------------------------------
expected_metadata_cols = ["filename", "class_label", "source_dataset", "subset"]

print("\nChecking metadata columns...")
for col in expected_metadata_cols:
    if col in df_train.columns:
        print(f"OK: {col}")
    else:
        raise ValueError(f"Missing required column: {col}")

# --------------------------------------------
# Verify feature count
# --------------------------------------------
feature_cols = [col for col in df_train.columns if col not in expected_metadata_cols]

print(f"\nNumber of feature columns: {len(feature_cols)}")

EXPECTED_FEATURE_COUNT = 25

if len(feature_cols) == EXPECTED_FEATURE_COUNT:
    print("Feature count is correct.")
else:
    raise ValueError(f"Expected {EXPECTED_FEATURE_COUNT} features, found {len(feature_cols)}.")

print("\nValidation complete.")



Performing sanity checks...

Total missing values: 0
No missing values detected.

Class distribution:
class_label
rl    7200
ai    7200
Name: count, dtype: int64

Checking metadata columns...
OK: filename
OK: class_label
OK: source_dataset
OK: subset

Number of feature columns: 25
Feature count is correct.

Validation complete.


In [6]:
# ============================================
# Cell 4: Prepare X_train and y_train
# ============================================

print("Preparing feature matrix X and target vector y...\n")

# -------------------------------------------------
# Define metadata columns
# -------------------------------------------------
metadata_cols = ["filename", "class_label", "source_dataset", "subset"]

# -------------------------------------------------
# Define feature columns
# -------------------------------------------------
feature_cols = [col for col in df_train.columns if col not in metadata_cols]

# -------------------------------------------------
# Build X and y
# -------------------------------------------------
X_train = df_train[feature_cols].values
y_train = df_train["class_label"].values

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")

# -------------------------------------------------
# Encode labels (rl, ai → 0, 1)
# -------------------------------------------------
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)

print("\nLabel encoding mapping:")
for i, label in enumerate(label_encoder.classes_):
    print(f"{label} → {i}")

# -------------------------------------------------
# Final confirmation
# -------------------------------------------------
print("\nEncoded class distribution:")
unique, counts = np.unique(y_train_encoded, return_counts=True)
for u, c in zip(unique, counts):
    print(f"Class {u}: {c}")

print("\nPreparation complete.")



Preparing feature matrix X and target vector y...

X_train shape: (14400, 25)
y_train shape: (14400,)

Label encoding mapping:
ai → 0
rl → 1

Encoded class distribution:
Class 0: 7200
Class 1: 7200

Preparation complete.


In [7]:
# ============================================
# Cell 5: Define Candidate Classifiers
# ============================================

print("Defining candidate classifiers and scoring metrics...\n")

# -------------------------------------------------
# Define cross-validation strategy
# -------------------------------------------------
cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# -------------------------------------------------
# Define candidate classifiers
# -------------------------------------------------
candidate_classifiers = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "Linear SVM": SVC(
        kernel="linear",
        probability=True,
        random_state=42
    ),

    "RBF SVM": SVC(
        kernel="rbf",
        probability=True,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),

    "Extra Trees": ExtraTreesClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        random_state=42
    ),

    "KNN": KNeighborsClassifier(
        n_neighbors=5
    ),

    "Gaussian NB": GaussianNB(),

    "MLP": MLPClassifier(
        hidden_layer_sizes=(64, 32),
        max_iter=300,
        random_state=42
    )
}

# -------------------------------------------------
# Define scoring metrics
# -------------------------------------------------
scoring_metrics = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

print("Candidate classifiers:")
for name in candidate_classifiers.keys():
    print(f" - {name}")

print("\nScoring metrics:")
for metric in scoring_metrics.keys():
    print(f" - {metric}")

print("\nCross-validation strategy:")
print(" - StratifiedKFold")
print(" - n_splits = 5")
print(" - shuffle = True")
print(" - random_state = 42")

print("\nClassifier setup complete.")



Defining candidate classifiers and scoring metrics...

Candidate classifiers:
 - Logistic Regression
 - Linear SVM
 - RBF SVM
 - Random Forest
 - Extra Trees
 - Gradient Boosting
 - KNN
 - Gaussian NB
 - MLP

Scoring metrics:
 - accuracy
 - precision
 - recall
 - f1
 - roc_auc

Cross-validation strategy:
 - StratifiedKFold
 - n_splits = 5
 - shuffle = True
 - random_state = 42

Classifier setup complete.


In [9]:
# ============================================
# Cell 6: Run Baseline Classifier Comparison
# ============================================

print("Running baseline classifier comparison...\n")

# NOTE:
# Execution time for this cell can be significantly reduced (often ~2× faster)
# by selecting a High-RAM runtime in Colab:
# Runtime → Change runtime type → Runtime shape → High-RAM
#
# This is especially beneficial due to parallel cross-validation (n_jobs=-1)
# and memory-intensive models such as SVM and MLP.

import time

baseline_results = []
total_models = len(candidate_classifiers)

for i, (name, clf) in enumerate(candidate_classifiers.items(), 1):
    print(f"[{i}/{total_models}] Evaluating: {name}")

    start_time = time.time()

    cv_results = cross_validate(
        estimator=clf,
        X=X_train,
        y=y_train_encoded,
        cv=cv_strategy,
        scoring=scoring_metrics,
        return_train_score=False,
        n_jobs=-1
    )

    elapsed = time.time() - start_time

    result = {
        "classifier": name,
        "accuracy_mean": np.mean(cv_results["test_accuracy"]),
        "accuracy_std": np.std(cv_results["test_accuracy"]),
        "precision_mean": np.mean(cv_results["test_precision"]),
        "precision_std": np.std(cv_results["test_precision"]),
        "recall_mean": np.mean(cv_results["test_recall"]),
        "recall_std": np.std(cv_results["test_recall"]),
        "f1_mean": np.mean(cv_results["test_f1"]),
        "f1_std": np.std(cv_results["test_f1"]),
        "roc_auc_mean": np.mean(cv_results["test_roc_auc"]),
        "roc_auc_std": np.std(cv_results["test_roc_auc"])
    }

    baseline_results.append(result)

    print(f"Completed: {name} (time: {elapsed:.2f} sec)\n")

print("Baseline classifier comparison complete.")



Running baseline classifier comparison...

[1/9] Evaluating: Logistic Regression
Completed: Logistic Regression (time: 0.25 sec)

[2/9] Evaluating: Linear SVM
Completed: Linear SVM (time: 36.60 sec)

[3/9] Evaluating: RBF SVM
Completed: RBF SVM (time: 31.20 sec)

[4/9] Evaluating: Random Forest
Completed: Random Forest (time: 5.84 sec)

[5/9] Evaluating: Extra Trees
Completed: Extra Trees (time: 1.67 sec)

[6/9] Evaluating: Gradient Boosting
Completed: Gradient Boosting (time: 14.48 sec)

[7/9] Evaluating: KNN
Completed: KNN (time: 0.53 sec)

[8/9] Evaluating: Gaussian NB
Completed: Gaussian NB (time: 0.14 sec)

[9/9] Evaluating: MLP
Completed: MLP (time: 14.48 sec)

Baseline classifier comparison complete.


In [10]:
# ============================================
# Cell 7: Compile and Rank Results
# ============================================

print("Compiling and ranking baseline classifier results...\n")

# -------------------------------------------------
# Convert results to DataFrame
# -------------------------------------------------
df_baseline_results = pd.DataFrame(baseline_results)

# -------------------------------------------------
# Sort by primary metric (ROC-AUC)
# -------------------------------------------------
df_baseline_results = df_baseline_results.sort_values(
    by="roc_auc_mean",
    ascending=False
).reset_index(drop=True)

# -------------------------------------------------
# Display results
# -------------------------------------------------
print("Baseline classifier comparison (sorted by ROC-AUC):\n")
display(df_baseline_results)

# -------------------------------------------------
# Display top classifiers
# -------------------------------------------------
TOP_K = 3

print(f"\nTop {TOP_K} classifiers based on ROC-AUC:\n")

for i in range(min(TOP_K, len(df_baseline_results))):
    row = df_baseline_results.iloc[i]
    print(
        f"{i+1}. {row['classifier']} | "
        f"ROC-AUC: {row['roc_auc_mean']:.4f} | "
        f"F1: {row['f1_mean']:.4f} | "
        f"Accuracy: {row['accuracy_mean']:.4f}"
    )

print("\nRanking complete.")



Compiling and ranking baseline classifier results...

Baseline classifier comparison (sorted by ROC-AUC):



,classifier,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std
0,RBF SVM,0.777778,0.005835,0.773080,0.005561,0.786389,0.007962,0.779665,0.006092,0.861125,0.005117
1,MLP,0.776042,0.006559,0.769598,0.014522,0.789167,0.025072,0.778821,0.008275,0.859080,0.008704
2,Random Forest,0.766111,0.005287,0.771078,0.007182,0.757083,0.008361,0.763974,0.005326,0.847194,0.006478
3,Extra Trees,0.756806,0.003456,0.756472,0.004165,0.757500,0.006202,0.756967,0.003734,0.842692,0.006798
4,Gradient Boosting,0.750278,0.007898,0.747784,0.010058,0.755556,0.009986,0.751591,0.007489,0.830625,0.007015
5,Linear SVM,0.727153,0.005853,0.712224,0.005775,0.762361,0.007289,0.736428,0.005756,0.799640,0.009323
6,Logistic Regression,0.725833,0.004809,0.714878,0.004928,0.751389,0.009118,0.732647,0.005336,0.799256,0.009348
7,KNN,0.727153,0.001957,0.718349,0.002187,0.747361,0.007849,0.732539,0.003258,0.793381,0.005663
8,Gaussian NB,0.666667,0.005560,0.656834,0.006766,0.698333,0.011936,0.676869,0.005916,0.712219,0.006683



Top 3 classifiers based on ROC-AUC:

1. RBF SVM | ROC-AUC: 0.8611 | F1: 0.7797 | Accuracy: 0.7778
2. MLP | ROC-AUC: 0.8591 | F1: 0.7788 | Accuracy: 0.7760
3. Random Forest | ROC-AUC: 0.8472 | F1: 0.7640 | Accuracy: 0.7661

Ranking complete.


In [19]:
# ============================================
# Cell 8: Tune Top Classifiers
# ============================================

print("Tuning top 3 classifiers...\n")

import time
from itertools import product

tuned_results = []

# -------------------------------------------------
# Define parameter grids (small, controlled)
# -------------------------------------------------
param_grids = {
    "RBF SVM": {
        "C": [0.1, 1, 10],
        "gamma": ["scale", 0.1, 0.01]
    },

    "MLP": {
        "hidden_layer_sizes": [(64, 32), (128, 64, 32)],
        "alpha": [0.0001, 0.001],
        "learning_rate_init": [0.001, 0.0005]
    },

    "Random Forest": {
        "n_estimators": [100, 200],
        "max_depth": [None, 10, 20]
    }
}

# -------------------------------------------------
# Tune each top classifier
# -------------------------------------------------
top_classifiers = ["RBF SVM", "MLP", "Random Forest"]
total_models = len(top_classifiers)

for i, name in enumerate(top_classifiers, 1):
    print(f"[{i}/{total_models}] Tuning: {name}")

    clf = candidate_classifiers[name]
    param_grid = param_grids[name]

    # Count parameter combinations
    n_combinations = len(list(product(*param_grid.values())))
    print(f"Parameter combinations: {n_combinations}")

    start_time = time.time()

    grid_search = GridSearchCV(
        estimator=clf,
        param_grid=param_grid,
        scoring="roc_auc",
        cv=cv_strategy,
        verbose=1,
        n_jobs=-1
    )

    grid_search.fit(X_train, y_train_encoded)

    elapsed = time.time() - start_time

    best_model = grid_search.best_estimator_
    best_score = grid_search.best_score_

    print(f"Best ROC-AUC: {best_score:.4f}")
    print(f"Best params: {grid_search.best_params_}")
    print(f"Time: {elapsed:.2f} sec\n")

    tuned_results.append({
        "classifier": name,
        "best_roc_auc": best_score,
        "best_params": grid_search.best_params_
    })

print("Tuning complete.")



Tuning top 3 classifiers...

[1/3] Tuning: RBF SVM
Parameter combinations: 9
Fitting 5 folds for each of 9 candidates, totalling 45 fits
Best ROC-AUC: 0.8790
Best params: {'C': 10, 'gamma': 'scale'}
Time: 267.23 sec

[2/3] Tuning: MLP
Parameter combinations: 8
Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best ROC-AUC: 0.8669
Best params: {'alpha': 0.001, 'hidden_layer_sizes': (64, 32), 'learning_rate_init': 0.0005}
Time: 137.06 sec

[3/3] Tuning: Random Forest
Parameter combinations: 6
Fitting 5 folds for each of 6 candidates, totalling 30 fits
Best ROC-AUC: 0.8488
Best params: {'max_depth': None, 'n_estimators': 200}
Time: 50.28 sec

Tuning complete.


In [20]:
## ============================================
# Cell 9: Save Results
# ============================================

print("Saving classifier comparison results...\n")

# -------------------------------------------------
# Convert tuned results to DataFrame
# -------------------------------------------------
df_tuned_results = pd.DataFrame(tuned_results)

# -------------------------------------------------
# Sort tuned results by best ROC-AUC
# -------------------------------------------------
df_tuned_results = df_tuned_results.sort_values(
    by="best_roc_auc",
    ascending=False
).reset_index(drop=True)

# -------------------------------------------------
# Save baseline and tuned comparison tables
# -------------------------------------------------
df_baseline_results.to_csv(BASELINE_RESULTS_CSV, index=False)
df_tuned_results.to_csv(TUNED_RESULTS_CSV, index=False)

print(f"Saved baseline results: {BASELINE_RESULTS_CSV}")
print(f"Saved tuned results:    {TUNED_RESULTS_CSV}")

# -------------------------------------------------
# Save best tuned classifier summary to JSON
# -------------------------------------------------
best_row = df_tuned_results.iloc[0]

best_classifier_summary = {
    "best_classifier": best_row["classifier"],
    "best_roc_auc": float(best_row["best_roc_auc"]),
    "best_params": best_row["best_params"]
}

with open(BEST_CLASSIFIER_JSON, "w") as f:
    json.dump(best_classifier_summary, f, indent=4)

print(f"Saved best classifier summary: {BEST_CLASSIFIER_JSON}")

# -------------------------------------------------
# Display saved results
# -------------------------------------------------
print("\nBest tuned classifier summary:")
print(best_classifier_summary)

print("\nSave complete.")



Saving classifier comparison results...

Saved baseline results: /content/drive/MyDrive/DIP_Project/models/classifier_comparison_baseline.csv
Saved tuned results:    /content/drive/MyDrive/DIP_Project/models/classifier_comparison_tuned.csv
Saved best classifier summary: /content/drive/MyDrive/DIP_Project/models/best_classifier_config.json

Best tuned classifier summary:
{'best_classifier': 'RBF SVM', 'best_roc_auc': 0.8789567901234567, 'best_params': {'C': 10, 'gamma': 'scale'}}

Save complete.


In [21]:
# ============================================
# Cell 10: Summary and Recommendations
# ============================================

print("Summarizing classifier selection results...\n")

# -------------------------------------------------
# Identify best baseline and tuned classifiers
# -------------------------------------------------
best_baseline_row = df_baseline_results.iloc[0]
best_tuned_row = df_tuned_results.iloc[0]

print("Best baseline classifier:")
print(f"  Classifier: {best_baseline_row['classifier']}")
print(f"  ROC-AUC:    {best_baseline_row['roc_auc_mean']:.4f}")
print(f"  F1 Score:   {best_baseline_row['f1_mean']:.4f}")
print(f"  Accuracy:   {best_baseline_row['accuracy_mean']:.4f}")

print("\nBest tuned classifier:")
print(f"  Classifier: {best_tuned_row['classifier']}")
print(f"  ROC-AUC:    {best_tuned_row['best_roc_auc']:.4f}")
print(f"  Parameters: {best_tuned_row['best_params']}")

# -------------------------------------------------
# Recommendation for subsequent notebook(s)
# -------------------------------------------------
recommended_classifier = best_tuned_row["classifier"]

print("\nRecommendation:")
print(
    f"  The recommended classifier for subsequent final training "
    f"and independent test evaluation is: {recommended_classifier}"
)

# -------------------------------------------------
# Interpretation
# -------------------------------------------------
print("\nInterpretation:")
print(
    "  Classifier selection was performed using the normalized "
    "25-feature DIP representation and stratified 5-fold cross-validation "
    "on the training set only. The held-out test set was not used in this "
    "notebook and remains reserved for final independent evaluation."
)

print(
    "\n  The baseline comparison identified the strongest classifier "
    "candidates, and light hyperparameter tuning was then applied to the "
    "top-performing models. The recommended classifier is the tuned model "
    "with the highest cross-validated ROC-AUC."
)

print("\nClassifier selection notebook complete.")



Summarizing classifier selection results...

Best baseline classifier:
  Classifier: RBF SVM
  ROC-AUC:    0.8611
  F1 Score:   0.7797
  Accuracy:   0.7778

Best tuned classifier:
  Classifier: RBF SVM
  ROC-AUC:    0.8790
  Parameters: {'C': 10, 'gamma': 'scale'}

Recommendation:
  The recommended classifier for subsequent final training and independent test evaluation is: RBF SVM

Interpretation:
  Classifier selection was performed using the normalized 25-feature DIP representation and stratified 5-fold cross-validation on the training set only. The held-out test set was not used in this notebook and remains reserved for final independent evaluation.

  The baseline comparison identified the strongest classifier candidates, and light hyperparameter tuning was then applied to the top-performing models. The recommended classifier is the tuned model with the highest cross-validated ROC-AUC.

Classifier selection notebook complete.
